In [40]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error


In [49]:
df = pd.read_csv("NFLAnalysis.csv")

#Clean up data
df.columns = df.columns.str.strip()
df = df.replace("—", np.nan)
df = df.replace({',': ''}, regex=True)

for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass


In [50]:
df.head()

,Player,Age,School,Height,Weight,Arm,40-yd,10-yd,Shuttle,3-cone,Vert,Broad,Bench,C Receptions,C Yards,C TDs,N Receptions,N Yards,N TDs
0,Jordan Addison,21,P4,71,173,30.85,4.49,1.56,NaN,NaN,34.0,1020.0,NaN,219,3134,29,180,2468,22
1,Ronnie Bell,23,P4,72,191,31.00,4.54,1.52,4.15,6.98,38.5,1000.0,14.0,145,2269,9,13,120,0
2,Jake Bobo,23,P4,76,206,32.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,183,2258,10,26,308,2
3,Kayshon Boutte,21,P4,71,195,31.35,4.50,1.58,4.25,NaN,29.0,910.0,NaN,131,1782,16,7,71,0
4,Jalen Brooks,23,P4,73,201,34.15,4.69,1.56,4.31,7.15,35.0,1010.0,NaN,58,785,2,20,215,1


In [51]:
df.tail()

,Player,Age,School,Height,Weight,Arm,40-yd,10-yd,Shuttle,3-cone,Vert,Broad,Bench,C Receptions,C Yards,C TDs,N Receptions,N Yards,N TDs
45,Tre Tucker,22,P4,69,182,28.85,4.40,1.48,NaN,NaN,37.5,1040.0,NaN,97,1424,11,62,805,3
46,Parker Washington,20,P4,70,204,29.00,NaN,NaN,NaN,NaN,NaN,NaN,16.0,131,1920,12,0,0,0
47,Jalen Wayne,21,G5,74,210,32.15,4.51,1.54,NaN,NaN,34.5,1040.0,NaN,179,2377,13,0,0,0
48,Dontayvion Wicks,22,P4,73,206,32.30,4.62,1.59,NaN,NaN,39.0,1000.0,NaN,116,1816,14,58,698,9
49,Michael Wilson,22,G5,74,213,31.00,4.58,1.50,4.27,NaN,37.5,1050.0,23.0,132,1692,15,79,855,3


In [44]:
#These are the variables we use to make the prediction
feature_cols = [
    "Age","Height","Weight","Arm",
    "40-yd","10-yd","Shuttle","3-cone",
    "Vert","Broad","Bench",
    "C Receptions","C Yards","C TDs"]

X = df[feature_cols]

#Predicting NFL production
targets = ["N Yards", "N Receptions", "N TDs"]
models = {}

In [45]:
for t in targets:
    y = df[t]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("gbr", GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)
    models[t] = model

    preds = model.predict(X_test)
    print(f"\nModel for {t}")
    print("R²:", round(r2_score(y_test, preds), 3))
    print("MAE:", round(mean_absolute_error(y_test, preds), 1))



Model for N Yards
R²: -1.029
MAE: 568.0

Model for N Receptions
R²: -0.654
MAE: 41.1

Model for N TDs
R²: 0.178
MAE: 3.4


In [46]:
def get_user_prospect():
    print("\nEnter new prospect data (leave blank if unknown):")
    
    data = {}
    for col in feature_cols:
        val = input(f"{col}: ")
        data[col] = float(val) if val != "" else np.nan
    
    return pd.DataFrame([data])

new_player = get_user_prospect()



Enter new prospect data (leave blank if unknown):


Age:  21
Height:  72
Weight:  187
Arm:  31
40-yd:  4.39
10-yd:  1.5
Shuttle:  
3-cone:  
Vert:  32
Broad:  1004
Bench:  15
C Receptions:  176
C Yards:  2711
C TDs:  35


In [47]:
pred_yards = models["N Yards"].predict(new_player)[0]
pred_recs  = models["N Receptions"].predict(new_player)[0]
pred_tds   = models["N TDs"].predict(new_player)[0]

print("\n--- NFL Projection (First 3 Years) ---")
print("Projected Yards:", int(pred_yards))
print("Projected Receptions:", int(pred_recs))
print("Projected TDs:", int(pred_tds))



--- NFL Projection (First 3 Years) ---
Projected Yards: 1406
Projected Receptions: 117
Projected TDs: 15
